# Threshold review in a live kernel
Use the Academy fraud lab to generate evaluation.csv from the synthetic held-out sample.
Cell identity, execution count, saved source and Run identity answer separate questions.


In [ ]:
import csv
from pathlib import Path

from apizr import tracking

root = Path.cwd()
calibration_offset = 0.0
with Path("evaluation.csv").open() as stream:
    rows = list(csv.DictReader(stream))


def evaluate(threshold):
    actual = [int(row["fraud"]) for row in rows]
    predicted = [
        min(1.0, float(row["probability"]) + calibration_offset) >= threshold
        for row in rows
    ]
    tp = sum(bool(y) and p for y, p in zip(actual, predicted, strict=True))
    fp = sum(not y and p for y, p in zip(actual, predicted, strict=True))
    fn = sum(bool(y) and not p for y, p in zip(actual, predicted, strict=True))
    return {
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "flagged": sum(predicted),
    }

In [ ]:
with tracking.session(root=root) as capture:
    tracking.log_input("holdout", "evaluation.csv")
    tracking.log_parameter("threshold", 0.5)
    result = evaluate(0.5)
    for name, value in result.items():
        tracking.log_metric(name, value)
print("Session result", result)
print("Captured cell", capture.record.run.capture.cell.execution_count)

In [ ]:
@tracking.experiment(parameters=["threshold"], root=root)
def tracked_evaluate(threshold):
    tracking.log_input("holdout", "evaluation.csv")
    result = evaluate(threshold)
    for name, value in result.items():
        tracking.log_metric(name, value)
    return result

In [ ]:
first = tracked_evaluate(0.35)
print("First threshold", first)

In [ ]:
second = tracked_evaluate(0.65)
print("Second threshold", second)

In [ ]:
# Intentional advanced variant: this live setting is NOT logged.
calibration_offset = 0.2
print("Unlogged calibration offset changed; repeat first-variant next.")